# Efektywne programowanie w języku Python
## Wykład 6 — mierzenie, profilowanie i kompilacja

### Kolejność decyzji
**Poprawność → algorytm → pomiar → biblioteka / NumPy → kompilacja → równoległość.**

Kompilator nie naprawia złego algorytmu. Procesy z `part05` także mają narzut.
Porównujemy ten sam wynik, dane i zasady pomiaru, nie historyczne rankingi narzędzi.

### Przygotowanie środowiska
Przykłady sprawdzamy dla Pythona 3.10 i NumPy 1.26.2.
Przed wykładem, w terminalu środowiska kursu:
```text
python -m pip install -r part06_files/requirements.txt
```
[Lista dodatków](part06_files/requirements.txt): Cython 3.0.12, Numba 0.60.0, line_profiler 4.2.0.
Notebook nie instaluje pakietów przy „Run All”.

### Cython potrzebuje kompilatora
Cython tłumaczy kod na C, a kompilator buduje rozszerzenie.
Na Windows potrzebne są zgodne narzędzia C/C++ (np. MSVC Build Tools).

Numba korzysta z LLVM przez llvmlite. Komórki Cythona wymagają przygotowanego kompilatora; samego pakietu `Cython` nie wystarczy.

In [ ]:
import sys
import numpy as np
from functools import cache
from time import perf_counter
from statistics import median
import Cython, numba, line_profiler
print("Python", sys.version.split()[0], "NumPy", np.__version__)
print("Cython", Cython.__version__, "Numba", numba.__version__)

## 1. Algorytm przed kompilatorem
Fibonacci jest demonstracją powtarzania obliczeń, a nie praktycznym benchmarkiem wszystkich zastosowań Pythona.

Kontrakt: całkowite `n >= 0`. Rekurencję naiwną uruchamiamy tylko dla małego `n`.

In [ ]:
def validate_n(n):
    if type(n) is not int or n < 0:
        raise ValueError("n musi być nieujemnym int")

def fib_recursive(n):
    validate_n(n)
    def calculate(k):
        if k < 2:
            return k
        return calculate(k - 1) + calculate(k - 2)
    return calculate(n)

assert fib_recursive(20) == 6765

In [ ]:
def fib_iterative(n):
    validate_n(n)
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

@cache
def fib_cached(n):
    validate_n(n)
    if n < 2:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

In [ ]:
expected_fib = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
assert [fib_iterative(i) for i in range(11)] == expected_fib
assert [fib_cached(i) for i in range(11)] == expected_fib
assert [fib_recursive(i) for i in range(11)] == expected_fib
for invalid in [-1, 1.5, True]:
    try:
        fib_iterative(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Brak walidacji")

### Złożoność i ograniczenia
Rekurencja naiwna wykonuje wykładniczo wiele wywołań. Cache usuwa powtarzane obliczenia, ale zajmuje pamięć i nie usuwa limitu rekurencji.

Iteracja wykonuje liniowo wiele kroków i przechowuje dwie liczby. To nie stała liczba bajtów: Pythonowe `int` rosną wraz z wynikiem, a koszt ich dodawania także rośnie.

### Co mierzy `%timeit`?
`-n` ustawia liczbę wykonań w serii, `-r` liczbę serii.
Bez jawnego `-n` IPython dobiera liczbę iteracji. Standardowy raport podaje średnią i odchylenie.

`-o` zwraca obiekt z wynikami. Nie zakładamy zawsze „100 wywołań, najlepszy z trzech”.

In [ ]:
recursive_timing = %timeit -n 10 -r 3 -o fib_recursive(20)
iterative_timing = %timeit -n 1000 -r 3 -o fib_iterative(20)

### Cache: zimny i rozgrzany
Zimny wariant zaczyna bez zapamiętanych wyników. Ciepły wariant mierzy trafienie w cache, a nie ponowne obliczenie Fibonacciego.

To dwa różne scenariusze. Czyszczenie cache poniżej jest częścią mierzonego zimnego wywołania.

In [ ]:
def cold_fib():
    fib_cached.cache_clear()
    return fib_cached(20)

assert cold_fib() == 6765
cold_timing = %timeit -n 100 -r 3 -o cold_fib()
fib_cached(20)
warm_timing = %timeit -n 1000 -r 3 -o fib_cached(20)
print(fib_cached.cache_info())

### Koszt cache nie kończy się na czasie
`cache_info()` pokazuje trafienia, chybienia i rozmiar.
`cache_clear()` usuwa stan. `functools.cache` jest dostępny od Pythona 3.9; backport nie jest potrzebny.

Cache ma sens dla powtarzalnych argumentów i stabilnych wyników. Nie rozwiązuje automatycznie problemów mutowalności, efektów ubocznych i pamięci.

### Rozszerzenie: cache bez trafień
Czy zapamiętywanie wyników zawsze pomaga?
Dla tej samej funkcji porównaj powtarzanie argumentu z podawaniem nowych argumentów.

Celowo wybieramy bardzo tanie obliczenie. Sprawdzimy statystyki i liczbę wpisów, bez obietnicy przyspieszenia.

In [ ]:
@cache
def cached_square(number):
    return number * number

cached_square.cache_clear()
repeated_results = [cached_square(7) for _ in range(1000)]
repeated_info = cached_square.cache_info()
assert repeated_info.hits == 999 and repeated_info.currsize == 1
cached_square.cache_clear()
unique_results = [cached_square(i) for i in range(1000)]
unique_info = cached_square.cache_info()
assert unique_info.hits == 0 and unique_info.currsize == 1000
assert unique_results == [i * i for i in range(1000)]
print("Powtórzenia:", repeated_info)
print("Nowe argumenty:", unique_info)
cached_square.cache_clear()

### Rozszerzenie: co wynika z tego eksperymentu?
`currsize` liczy wpisy, a nie bajty. Cache przechowuje referencje do argumentów i wyników; bez limitu może rosnąć.

Dla strumienia nowych argumentów nie ma trafień, ale pozostaje koszt obsługi i przechowywania. Rozważ ograniczone `lru_cache(maxsize=...)` albo brak cache.
Nie przenoś tego wniosku na kosztowne obliczenia bez pomiaru.

## 2. Profilowanie przed optymalizacją
Oddzielamy generowanie danych od konwersji `int → hex → int`.
Wszystkie warianty dostają te same 1000 liczb z tego samego zakresu.

`2**31 - 1` to potęgowanie; `2 ^ 31 - 1` to XOR i daje 28.

In [ ]:
rng = np.random.default_rng(100)
values = rng.integers(0, 2**31 - 1, size=1000, dtype=np.int32)
python_values = values.tolist()
assert (2 ^ 31 - 1) == 28

def parse_values(numbers):
    return [int(hex(value), 16) for value in numbers]

assert parse_values(python_values) == python_values
assert parse_values([0, -1, 2**40]) == [0, -1, 2**40]

### `cProfile`: gdzie program spędza czas?
Profilowanie służy do szukania kosztownych miejsc. Jego narzut oznacza, że nie zastępuje końcowego benchmarku.

Zaczynamy od narzędzia biblioteki standardowej. Używamy krótkiego raportu zamiast wielkiego wydruku na slajdzie.

In [ ]:
import cProfile
import pstats
import io

profile = cProfile.Profile()
profile.runcall(parse_values, python_values)
report = io.StringIO()
pstats.Stats(profile, stream=report).strip_dirs().sort_stats("cumulative").print_stats(6)
print(report.getvalue())

### Rozszerzenie: profilowanie wierszy
`line_profiler` pokazuje rozkład czasu wewnątrz funkcji.
Aby rozdzielić etapy, zapisujemy pętlę w kilku wierszach.
Nie wyciągamy wniosku o losowaniu z funkcji, która już go nie zawiera.

In [ ]:
def parse_loop(numbers):
    output = []
    for value in numbers:
        text = hex(value)
        parsed = int(text, 16)
        output.append(parsed)
    return output

assert parse_loop(python_values) == python_values

In [ ]:
%load_ext line_profiler
%lprun -f parse_loop parse_loop(python_values)

### `np.vectorize()` nie kompiluje pętli
To wygodny interfejs stosujący funkcję Pythona do elementów z broadcastingiem.
Nie należy utożsamiać go z natywnymi operacjami tablicowymi.

Podajemy typy wyjściowe, aby uniknąć ich zgadywania na podstawie pierwszego elementu.

In [ ]:
vhex = np.vectorize(hex, otypes=[str])
vint = np.vectorize(lambda text: int(text, 16), otypes=[np.int64])

def parse_vectorize(numbers):
    return vint(vhex(numbers))

np.testing.assert_array_equal(parse_vectorize(values), values)
%timeit -n 10 -r 3 parse_values(python_values)
%timeit -n 10 -r 3 parse_vectorize(values)

## 3. Numba: kompilacja kodu numerycznego
`@njit` jawnie wymaga trybu nopython. Importujemy publiczne API: `from numba import njit`.

Pierwsze wywołanie dla nowych typów argumentów może kompilować kod. Osobno mierzymy rozruch i kolejne wywołania.

In [ ]:
from numba import njit

@njit
def fib_numba_core(n):
    a, b = 0, 1
    for i in range(n):
        if i == n - 1:
            return b
        a, b = b, a + b
    return a

start = perf_counter()
numba_first = fib_numba_core(20)
numba_startup = perf_counter() - start
assert numba_first == fib_iterative(20)
print(f"Pierwsze wywołanie, w tym kompilacja: {numba_startup:.6f} s")

### Zmiana reprezentacji zmienia kontrakt
Numba używa tutaj liczb o stałej szerokości. Cythonowe `long` jest zależne od platformy — na Windows zwykle ma 32 bity.

Dla porównywalności przykłady skompilowane przyjmują tylko `0 <= n <= 92` i używają wyniku 64-bitowego. Dla większych wyników zostaje Pythonowy `int`.
Same adnotacje `n: int` w zwykłym Pythonie nie zapewniają kompilacji.

In [ ]:
def fib_numba(n):
    validate_n(n)
    if n > 92:
        raise ValueError("Zakres demonstracji: 0..92")
    return fib_numba_core(n)

assert fib_numba(92) == fib_iterative(92)
%timeit -n 1000 -r 3 fib_numba(20)

### Rozszerzenie: granice kompilatora
Numba nie kompiluje dowolnego kodu Pythonowego. Celowo próbujemy skompilować operację `hex()`.

W sprawdzanej wersji 0.60 oczekujemy `TypingError`. To błąd typowania kompilatora, a nie błędny wynik obliczenia. Po aktualizacji pakietu przykład wymaga ponownej kontroli.

In [ ]:
from numba.core.errors import TypingError

@njit
def unsupported_hex(value):
    return hex(value)

try:
    unsupported_hex(10)
except TypingError:
    print("Oczekiwany TypingError: hex poza obsługiwanym podzbiorem")
else:
    print("Ta wersja kompilatora obsługuje już przykład — sprawdź materiał")

## 4. Cython: jawne typy i kod natywny
`def` udostępnia funkcję Pythonowi; `cdef` definiuje funkcję Cython/C; `cpdef` daje także interfejs Pythonowy.

Parametry i zmienne można typować również w `def`. Samo zastąpienie słowa kluczowego nie jest uniwersalną optymalizacją.

In [ ]:
import shutil
import platform

if platform.system() == "Windows":
    from setuptools._distutils._msvccompiler import _get_vc_env
    try:
        _get_vc_env("x64")
        has_c_compiler = True
    except Exception as error:
        has_c_compiler = False
        print("Cython pominięty: brak dostępnego MSVC Build Tools")
else:
    has_c_compiler = bool(shutil.which("cc") or shutil.which("gcc") or shutil.which("clang"))
if has_c_compiler:
    get_ipython().run_line_magic("load_ext", "Cython")
else:
    print("Kod Cythona pozostaje materiałem do wykonania po przygotowaniu kompilatora")

In [ ]:
cython_source = r'''
cpdef long long fib_cython(int n):
    if n < 0 or n > 92:
        raise ValueError("Zakres demonstracji: 0..92")
    cdef long long a = 0, b = 1, next_value
    cdef int i
    for i in range(n):
        # Przy n=92 nie obliczamy F(93), które przekracza int64.
        if i == n - 1:
            return b
        next_value = a + b
        a, b = b, next_value
    return a
'''
if has_c_compiler:
    get_ipython().run_cell_magic("cython", "", cython_source)
else:
    print("Pominięto kompilację Cythona — brak kompilatora C/C++")

In [ ]:
if has_c_compiler:
    for index in [0, 1, 2, 20, 92]:
        assert fib_cython(index) == fib_iterative(index)
    get_ipython().run_line_magic("timeit", "-n 1000 -r 3 fib_cython(20)")

### Co porównujemy w Fibonaccim?
Rekurencja i iteracja pokazują zmianę algorytmu. Cache pokazuje wykorzystanie wcześniejszej pracy.

Cython i Numba pokazują koszt wykonywania liniowej pętli w różnych reprezentacjach. Wrappers i walidacja też kosztują.
Nie łączymy tych wyników w ranking „najlepszego języka”.

## 5. Tablice: poprawność, alokacja i kompilacja
Najpierw zgodność kształtu i typu. Dla głównego przykładu używamy dwuwymiarowych tablic `float64`.

Oddzielny eksperyment pokaże pułapkę `uint8`.

In [ ]:
a = rng.random((100, 100))
b = rng.random((100, 100))
a_before, b_before = a.copy(), b.copy()

def check_arrays(a, b):
    if a.ndim != 2 or b.ndim != 2 or a.shape != b.shape:
        raise ValueError("Potrzebne dwie macierze tego samego kształtu")
    if a.dtype != np.float64 or b.dtype != np.float64:
        raise TypeError("Wymagany float64")

In [ ]:
def add_python(a, b):
    check_arrays(a, b)
    output = np.empty_like(a)
    for y in range(a.shape[0]):
        for x in range(a.shape[1]):
            output[y, x] = a[y, x] + b[y, x]
    return output

reference = a + b
np.testing.assert_array_equal(add_python(a, b), reference)

### Cython: typowane memoryviews
`double[:, :]` udostępnia elementy bufora jako liczby C. Do indeksów używamy `Py_ssize_t`.

Zostawiamy sprawdzanie granic. Nie wyłączamy go dekoratorem, zanim nie ustalimy kontraktu i nie sprawdzimy poprawności.

In [ ]:
cython_source = r'''
import numpy as np

def add_cython(double[:, :] a, double[:, :] b):
    if a.shape[0] != b.shape[0] or a.shape[1] != b.shape[1]:
        raise ValueError("Różne kształty")
    cdef Py_ssize_t y, x
    output = np.empty((a.shape[0], a.shape[1]), dtype=np.float64)
    cdef double[:, :] view = output
    for y in range(a.shape[0]):
        for x in range(a.shape[1]):
            view[y, x] = a[y, x] + b[y, x]
    return output
'''
if has_c_compiler:
    get_ipython().run_cell_magic("cython", "", cython_source)
else:
    print("Pominięto kompilację Cythona — brak kompilatora C/C++")

In [ ]:
@njit
def add_numba_core(a, b):
    output = np.empty_like(a)
    for y in range(a.shape[0]):
        for x in range(a.shape[1]):
            output[y, x] = a[y, x] + b[y, x]
    return output

def add_numba(a, b):
    check_arrays(a, b)
    return add_numba_core(a, b)

np.testing.assert_array_equal(add_numba(a, b), reference)
if has_c_compiler:
    np.testing.assert_array_equal(add_cython(a, b), reference)

In [ ]:
add_functions = [add_python, add_numba]
if has_c_compiler:
    add_functions.append(add_cython)
for shape in [(0, 0), (1, 1), (2, 3)]:
    left, right = np.ones(shape), np.ones(shape)
    for function in add_functions:
        np.testing.assert_array_equal(function(left, right), left + right)
for function in add_functions:
    try:
        function(np.ones((2, 3)), np.ones((3, 2)))
    except ValueError:
        pass
    else:
        raise AssertionError("Brak kontroli kształtu")
np.testing.assert_array_equal(a, a_before)
np.testing.assert_array_equal(b, b_before)

### Mała liczba bitów, inny wynik
`uint8` mieści wartości 0..255. Operacja tablicowa na tym typie nie tworzy automatycznie dowolnie dużych liczb.

Jeżeli potrzebujesz większego zakresu, zmień typ **przed** dodawaniem.

In [ ]:
small_a = np.array([250], dtype=np.uint8)
small_b = np.array([10], dtype=np.uint8)
wrapped = small_a + small_b
wide = small_a.astype(np.uint16) + small_b.astype(np.uint16)
assert wrapped.tolist() == [4]
assert wide.tolist() == [260]
print("uint8:", wrapped, "uint16:", wide)

### Pomiar tej samej pracy
Warianty poniżej tworzą nową tablicę. NumPy jest naturalnym punktem odniesienia dla zwykłego dodawania.

Kompilację Numby i Cythona wykonaliśmy wcześniej. Nie ogłaszamy zwycięzcy na podstawie historycznej tabeli.

In [ ]:
%timeit -n 3 -r 3 add_python(a, b)
if has_c_compiler:
    get_ipython().run_line_magic("timeit", "-n 100 -r 3 add_cython(a, b)")
%timeit -n 100 -r 3 add_numba(a, b)
%timeit -n 100 -r 3 a + b

### Alokacja: nowa tablica czy gotowy bufor?
`np.add(a, b, out=output)` ponownie wykorzystuje pamięć wyniku.
To inne warunki niż tworzenie nowego wyniku w każdym wywołaniu.

Nie nadpisujemy wejścia. Przy użyciu `out=a` trzeba uwzględnić zmianę danych w kolejnych pomiarach.

In [ ]:
output_buffer = np.empty_like(a)
np.add(a, b, out=output_buffer)
np.testing.assert_array_equal(output_buffer, reference)
%timeit -n 100 -r 3 np.add(a, b, out=output_buffer)

### Rozszerzenie: pułapka mutowania wejścia
```python
np.add(work, increment, out=work)
```
Każde powtórzenie zmienia `work`. Czy benchmark nadal mierzy operację na tych samych danych?

Dla aktualizowania stanu może to być celowe. Dla porównania funkcji na stałym wejściu zmienia warunki eksperymentu.

In [ ]:
original = np.array([1.0, 2.0, 3.0])
increment = np.ones_like(original)
work = original.copy()
np.add(work, increment, out=work)
first_result = work.copy()
np.add(work, increment, out=work)
assert np.array_equal(first_result, original + 1)
assert np.array_equal(work, original + 2)
print("Pierwsze wywołanie:", first_result)
print("Drugie wywołanie:", work)

### Rozszerzenie: poprawny punkt odniesienia
Dla stałych danych wykorzystaj **osobny bufor wyniku**. Jeżeli musisz modyfikować wejście, przywracaj je przed próbą i jasno opisz, czy koszt kopiowania wchodzi do pomiaru.

W przykładzie poniżej wejście jest stałe, a koszt alokacji bufora wyniku pozostaje poza pomiarem.

In [ ]:
fixed_output = np.empty_like(original)
for _ in range(3):
    np.add(original, increment, out=fixed_output)
    np.testing.assert_array_equal(fixed_output, [2, 3, 4])
np.testing.assert_array_equal(original, [1, 2, 3])
%timeit -n 100 -r 3 np.add(original, increment, out=fixed_output)

### Rozszerzenie: kilka operacji w jednym przebiegu
Dla `sum((a-b)**2)` wyrażenie NumPy może tworzyć tablice tymczasowe.
Skompilowana pętla może połączyć odejmowanie, potęgowanie i sumowanie.

To lepszy powód użycia kompilatora niż zastępowanie pojedynczej operacji `a+b`.

In [ ]:
@njit
def squared_distance(a, b):
    total = 0.0
    for y in range(a.shape[0]):
        for x in range(a.shape[1]):
            delta = a[y, x] - b[y, x]
            total += delta * delta
    return total

expected_distance = np.sum((a - b)**2)
assert np.isclose(squared_distance(a, b), expected_distance)
%timeit -n 100 -r 3 squared_distance(a, b)
%timeit -n 100 -r 3 np.sum((a - b)**2)

### Rozszerzenie: kiedy kompilacja się zwróci?
Jeżeli kompilacja kosztuje `C`, a oszczędność na wywołaniu to `P - J`, próg wynosi w przybliżeniu `C / (P - J)` dla `P > J`.

Gdy `P <= J`, ten model nie przewiduje zwrotu kosztu. Nowe typy mogą tworzyć nowe specjalizacje.
Sprawdź, czy przyspieszenie ma znaczenie dla całego programu, nie tylko mikropętli.

### Rozszerzenie: dokładność i pamięć
Zmiana kolejności sumowania może zmienić zaokrąglenia. Dla obliczeń float wybieramy uzasadnione tolerancje zamiast automatycznego wymagania identycznych bitów.

`fastmath`, brak kontroli indeksów i równoległe redukcje zmieniają założenia. Włączaj je dopiero po testach i pomiarach.

### Rozszerzenie: suma zależy od kolejności
Przewidź wynik dla `[1e16, 1, -1e16]`. Matematycznie suma wynosi 1, ale `float` ma ograniczoną precyzję.

Porównujemy jawne sumowanie od lewej z `math.fsum()`. Wynik wbudowanego `sum()` może zależeć od wersji Pythona — nie zakładamy, że używa identycznego algorytmu.

In [ ]:
import math

terms = [1e16, 1.0, -1e16]
def left_sum(numbers):
    total = 0.0
    for number in numbers:
        total += number
    return total

reordered = [terms[0], terms[2], terms[1]]
assert left_sum(terms) == 0.0
assert left_sum(reordered) == 1.0
assert math.fsum(terms) == math.fsum(reordered) == 1.0
print("Od lewej:", left_sum(terms), left_sum(reordered))
print("sum:", sum(terms), "fsum:", math.fsum(terms))

### Rozszerzenie: dokładność jest częścią kontraktu
`math.fsum()` ogranicza utratę dokładności przy sumowaniu. To nie dowód, że dowolne przestawianie operacji jest bezpieczne.

Ustal tolerancję i rodzaj danych przed pomiarem. Inna kolejność redukcji albo `fastmath` może zmieniać wynik. Szybsza funkcja niespełniająca wymagań nie jest poprawną optymalizacją.

### Rozszerzenie: ten sam rozmiar, inne strides
Tablica ciągła i widok co drugiego elementu mogą mieć ten sam kształt i wartości, lecz inny odstęp między elementami w pamięci.

Sprawdź `strides` i `C_CONTIGUOUS`. Kopia do układu ciągłego kosztuje czas i pamięć, ale może pomóc w wielokrotnych przebiegach.

In [ ]:
storage = rng.random(400_000)
strided = storage[::2]
contiguous = np.ascontiguousarray(strided)
np.testing.assert_array_equal(strided, contiguous)
assert not strided.flags.c_contiguous
assert contiguous.flags.c_contiguous
assert np.shares_memory(storage, strided)
assert not np.shares_memory(storage, contiguous)
print("Widok:", strided.shape, strided.strides)
print("Kopia:", contiguous.shape, contiguous.strides)
assert np.isclose(strided.sum(), contiguous.sum())

In [ ]:
strided_time = %timeit -n 20 -r 3 -o strided.sum()
contiguous_time = %timeit -n 20 -r 3 -o contiguous.sum()
copy_time = %timeit -n 20 -r 3 -o np.ascontiguousarray(strided)
saving = strided_time.average - contiguous_time.average
if saving > 0:
    print("Szacowany próg liczby przebiegów:", math.ceil(copy_time.average / saving))
else:
    print("W tym pomiarze kopia nie daje oszczędności na przebiegu")

### Rozszerzenie: kopiowanie też trzeba policzyć
Szacunek `czas_kopii / oszczędność_na_przebiegu` ma sens tylko dla dodatniej oszczędności i danych używanych wielokrotnie.

Małe różnice mogą być szumem. Wynik zależy od rozmiaru danych, pamięci cache procesora i operacji. Nie kopiuj automatycznie każdego widoku.

### Rozszerzenie: szybsza funkcja, prawie ten sam program
Fragment zajmuje 5% czasu. Przyspieszamy go dziesięciokrotnie, reszta pozostaje bez zmian.

Jaki będzie czas całego programu? Obliczamy model, zamiast symulować go niedokładnymi opóźnieniami `sleep()`.

In [ ]:
total_before = 100.0
fraction = 0.05
local_speedup = 10.0
unchanged = total_before * (1 - fraction)
optimized = total_before * fraction / local_speedup
total_after = unchanged + optimized
global_speedup = total_before / total_after
assert math.isclose(total_after, 95.5)
print(f"Czas: {total_before:.1f} -> {total_after:.1f}")
print(f"Przyspieszenie całości: {global_speedup:.3f}x")
print(f"Nawet bez kosztu fragmentu: najwyżej {1 / (1 - fraction):.3f}x")

### Rozszerzenie: wybór miejsca optymalizacji
To zastosowanie prawa Amdahla. Dziesięciokrotne przyspieszenie fragmentu daje tu około **1,047×** dla całości, czyli skraca czas o **4,5%**.

Profilowanie pomaga znaleźć fragment o znaczącym udziale czasu. Oceniaj również koszt implementacji, utrzymania i zużycie pamięci, a nie tylko wynik mikropomiaru.

## Pytania kontrolne
1. Co mierzy ciepły cache?
2. Dlaczego pierwsze wywołanie Numby jest szczególne?
3. Czy `np.vectorize()` kompiluje kod do C?
4. Kiedy zmiana `dtype` zmienia poprawność wyniku?
5. Dlaczego `out=` wymaga osobnego opisu benchmarku?

### Odpowiedzi
1. Ponowne użycie zapamiętanego wyniku.
2. Może zawierać kompilację specjalizacji.
3. Nie — wywołuje funkcję dla kolejnych elementów.
4. Gdy zmienia zakres, precyzję lub reguły arytmetyki.
5. Korzysta z przygotowanego bufora zamiast tworzyć nowy wynik.

### Self reading: pomiar i biblioteki
- [IPython: timeit](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit)
- [cProfile](https://docs.python.org/3.10/library/profile.html)
- [functools.cache](https://docs.python.org/3.10/library/functools.html#functools.cache)
- [NumPy: vectorize](https://numpy.org/doc/1.26/reference/generated/numpy.vectorize.html)
- [line_profiler](https://kernprof.readthedocs.io/en/latest/)

### Self reading: kompilacja
- [Numba 0.60: pierwsze kroki](https://numba.readthedocs.io/en/0.60.0/user/5minguide.html)
- [Cython: memoryviews](https://docs.cython.org/en/stable/src/userguide/memoryviews.html)
- [Cython: kompilacja](https://docs.cython.org/en/stable/src/userguide/source_files_and_compilation.html)
- [jupyterlab-rise](https://github.com/jupyterlab-contrib/rise)

Historyczne tabele czasów nie stanowią wyników dla środowiska kursu.

### Źródła archiwalne — opcjonalnie
- [Numba vs Cython (2012)](https://jakevdp.github.io/blog/2012/08/24/numba-vs-cython/) — działa, lecz API i pomiary są historyczne.
- [Materiał IBM](https://www.ibm.com/developerworks/community/blogs/jfp/entry/Python_Meets_Julia_Micro_Performance?lang=en)
- [Film 1](https://www.youtube.com/watch?v=SUf-ALvk3cU), [film 2](https://www.youtube.com/watch?v=NfnMJMkhDoQ)

Dostępność materiału IBM i filmów niepotwierdzona. Podstawowe źródła znajdują się na poprzednich slajdach.